# Model 2 — Mood / Instrument Auto-Tagger (MagnaTagATune)  🏷️

A multi-label CNN over log-mel spectrograms predicting the **top-50 tags** of
MagnaTagATune (mood words like *calm, happy, sad, dark, upbeat*; instrument and
genre words). These tags enrich the per-track *vibe* description and can refine
the planner's similarity term.

> **Optional / advanced.** Heavier than model 1. The app runs fine without it;
> integrate it by loading `models_weights/model_2_mood_tagger.pt` in a wrapper
> analogous to `app/vibe_model.py`.

## 1 · Dataset — MagnaTagATune

* Official: https://mirg.city.ac.uk/codeapps/the-magnatagatune-dataset
* Audio: `mp3.zip.001/.002/.003` (concatenate then unzip). Annotations:
  `annotations_final.csv` (tab-separated).

Place as:
```
datasets/mtat/audio/<subdir>/<clip>.mp3
datasets/mtat/annotations_final.csv
```
We keep the 50 most frequent tags (the standard MTAT-50 benchmark subset).

In [1]:
import os, sys, glob, random, time
import numpy as np
# make the project package importable from train_models/
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path: sys.path.insert(0, ROOT)
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import librosa
from app.vibe_model import GenreCNN, log_mel, GTZAN_GENRES, WEIGHTS_PATH, EMBED_DIM
from app.config import ANALYSIS_SR, N_MELS, MODELS_DIR
import pandas as pd
device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)

device mps


In [2]:
MTAT = os.path.join(ROOT, 'datasets', 'mtat')
ANNO = os.path.join(MTAT, 'annotations_final.csv')
AUDIO = os.path.join(MTAT, 'audio')
SR = ANALYSIS_SR; FRAMES = 128; BATCH = 32; EPOCHS = 20; LR = 1e-3
assert os.path.exists(ANNO), f'annotations not found at {ANNO} — see markdown.'
df = pd.read_csv(ANNO, sep='\t')
tag_cols = [c for c in df.columns if c not in ('clip_id', 'mp3_path')]
top = df[tag_cols].sum().sort_values(ascending=False).head(50).index.tolist()
print('top tags:', top[:12], '...')
df = df[df[top].sum(axis=1) > 0].reset_index(drop=True)
print(len(df), 'clips with >=1 top tag')

top tags: ['guitar', 'classical', 'slow', 'techno', 'strings', 'drums', 'electronic', 'rock', 'fast', 'piano', 'ambient', 'beat'] ...
21111 clips with >=1 top tag


In [3]:
from app.vibe_model import GenreCNN  # reuse architecture, swap head
class Tagger(nn.Module):
    def __init__(self, n_tags=50):
        super().__init__()
        base = GenreCNN(n_classes=n_tags)
        self.features, self.pool, self.embed = base.features, base.pool, base.embed
        self.head = nn.Sequential(nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, n_tags))
    def forward(self, x):
        h = self.pool(self.features(x)).flatten(1)
        return self.head(self.embed(h))

class MTAT(Dataset):
    def __init__(self, frame): self.df = frame
    def __len__(self): return len(self.df)
    def __getitem__(self, k):
        row = self.df.iloc[k]
        fp = os.path.join(AUDIO, row['mp3_path'])
        try: y,_ = librosa.load(fp, sr=SR, mono=True)
        except Exception: return self.__getitem__((k+1) % len(self.df))
        m = log_mel(y, SR); T = m.shape[1]
        s = random.randint(0, max(0, T-FRAMES))
        w = m[:, s:s+FRAMES]
        if w.shape[1] < FRAMES: w = np.pad(w, ((0,0),(0,FRAMES-w.shape[1])), mode='edge')
        lab = row[top].values.astype('float32')
        return torch.from_numpy(w).float(), torch.from_numpy(lab)

In [4]:
n_val = int(len(df)*0.1); vd = df.iloc[:n_val]; td = df.iloc[n_val:]
tdl = DataLoader(MTAT(td), batch_size=BATCH, shuffle=True)
vdl = DataLoader(MTAT(vd), batch_size=BATCH)
model = Tagger(50).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
crit = nn.BCEWithLogitsLoss()

In [5]:
from sklearn.metrics import roc_auc_score
OUT = MODELS_DIR / 'model_2_mood_tagger.pt'; best = 0.0
for ep in range(EPOCHS):
    model.train()
    for x,y in tdl:
        x=x.unsqueeze(1).to(device); y=y.to(device)
        opt.zero_grad(); loss=crit(model(x), y); loss.backward(); opt.step()
    model.eval(); P=[]; Y=[]
    with torch.no_grad():
        for x,y in vdl:
            P.append(torch.sigmoid(model(x.unsqueeze(1).to(device))).cpu().numpy()); Y.append(y.numpy())
    P=np.concatenate(P); Y=np.concatenate(Y)
    try: auc=roc_auc_score(Y, P, average='macro')
    except Exception: auc=float('nan')
    print(f'epoch {ep+1} val AUC {auc:.3f}')
    if auc>best:
        best=auc; torch.save({'state_dict':model.state_dict(),'tags':top,'auc':auc}, OUT)
print('best AUC', round(best,3), '->', OUT)

/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_4701/3613742855.py:18: UserWarning: PySoundFile failed. Trying audioread instead.
  try: y,_ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_4701/3613742855.py:18: UserWarning: PySoundFile failed. Trying audioread instead.
  try: y,_ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/var/folders/bq/2bdwpdy14s12c5jr2k5scc

epoch 1 val AUC 0.707


/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_4701/3613742855.py:18: UserWarning: PySoundFile failed. Trying audioread instead.
  try: y,_ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_4701/3613742855.py:18: UserWarning: PySoundFile failed. Trying audioread instead.
  try: y,_ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/var/folders/bq/2bdwpdy14s12c5jr2k5scc

epoch 2 val AUC 0.825


/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_4701/3613742855.py:18: UserWarning: PySoundFile failed. Trying audioread instead.
  try: y,_ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_4701/3613742855.py:18: UserWarning: PySoundFile failed. Trying audioread instead.
  try: y,_ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/var/folders/bq/2bdwpdy14s12c5jr2k5scc

...

epoch 18 val AUC 0.876


/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_4701/3613742855.py:18: UserWarning: PySoundFile failed. Trying audioread instead.
  try: y,_ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_4701/3613742855.py:18: UserWarning: PySoundFile failed. Trying audioread instead.
  try: y,_ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/var/folders/bq/2bdwpdy14s12c5jr2k5scc

epoch 19 val AUC 0.872


/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_4701/3613742855.py:18: UserWarning: PySoundFile failed. Trying audioread instead.
  try: y,_ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/var/folders/bq/2bdwpdy14s12c5jr2k5scclm0000gn/T/ipykernel_4701/3613742855.py:18: UserWarning: PySoundFile failed. Trying audioread instead.
  try: y,_ = librosa.load(fp, sr=SR, mono=True)
/Users/sam/Desktop/aiolearn/Apple/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/var/folders/bq/2bdwpdy14s12c5jr2k5scc

epoch 20 val AUC 0.873
best AUC 0.878 -> /Users/sam/Desktop/aiolearn/My_projects/AI_DJ/models_weights/model_2_mood_tagger.pt


## 3 · Using it

Load `model_2_mood_tagger.pt`, run `sigmoid(logits)` over several windows and
average to get per-clip tag probabilities. Mood tags (*calm/energetic/dark/
happy*) can be blended into the planner's vibe distance or surfaced in the UI.
Standard MTAT-50 macro-AUC for compact CNNs is ~0.88–0.90.